# OpenQARP — Official Tutorial 00: Hello qarpx (Colab Edition)

This notebook is the **first official tutorial** from the OpenQARP repository
(`examples/tutorial_00_hello_qarp.ipynb`), reproduced as-is from
https://github.com/OpenQARP/openqarp/tree/main/examples and wrapped with a
Colab install cell so it runs standalone, with nothing else changed.

OpenQARP numbers its tutorial series from `00`, so this is tutorial #1 in the
sequence — the smallest possible end-to-end example of the stack: build a
circuit, say what you want out of it, run it, read the result.

By Carlos Araque  
@KentryOps Data Labs  
Quantum researchers  
20-09-2026

## Setup (added for Colab — not part of the original tutorial)

Installs `openqarp` from PyPI. Wheels ship for Linux/macOS/Windows on Python
3.11–3.14, so this is a fast binary install on Colab's default runtime, no
compilation needed.

In [ ]:
!pip install -q openqarp

---

# Tutorial 00 — Hello qarpx

Welcome! This is the first notebook in a short series that walks you through
qarpx — the `qarp` package backed by the native C++ `qarpx` kernel.
We assume you know quantum computing; the series only teaches you *this stack*.

**The mental model** (one line per tutorial):

| Layer | Object | Role | Tutorial |
|---|---|---|---|
| Circuit | `Block` | what the quantum computer *does* | 01 |
| Quantity | `PrimitiveAlgorithm` (`Sampler`, `StateVector`, ...) | what you want to *extract* | 02 |
| Execution | `QarpEngine` | compiles once, runs many times | 03 |
| Loop | optimizers + composite algorithms (`VQE`, `QAOA`, ...) | full workflows | 04, 05 |

In this notebook we go through that whole stack once, in the smallest possible
example: build a Bell state and sample it.

In [ ]:
import qarp

qarp.__version__

## 1. Build a circuit

Circuits are `Block`s. A `SimpleBlock` is a flat list of gates with builder
methods (`.h()`, `.cx()`, `.rx()`, ...). You declare the number of qubits up
front, add gates, then call `.build()` — building finalises the block so the
engine can compile it.

In [ ]:
from qarp.blocks import SimpleBlock

bell = SimpleBlock(2, name="bell")
bell.h(0)
bell.cx(0, 1)
bell.measure([(q, q) for q in range(2)])  # measure qubit q into classical bit q
bell.build()

bell.plot()

## 2. Say what you want to compute

A *primitive* pairs a circuit with the quantity you want from it.
`Sampler` asks for the measured bitstring distribution.

In [ ]:
from qarp.algorithms import Sampler

sampler = Sampler(ket=bell, n_shots=4000)

## 3. Run it on the engine

`QarpEngine` compiles the primitive's circuits (`.build`) and simulates them
(`.run`). Passing `seed=` makes the shots reproducible.

In [ ]:
from qarp.engines import QarpEngine

engine = QarpEngine(seed=42)
engine.build([sampler])
results = engine.run()

distribution = results[0]
distribution

## 4. Read the result

You should see (up to shot noise) 50% `(0, 0)` and 50% `(1, 1)` — a Bell state.

Two things to internalise right away:

* The keys are **tuples of bits indexed by qubit**: position `q` in the tuple is
  qubit `q`. qarpx is **LSB-first** everywhere (qubit 0 = least-significant bit).
  Tutorial 01 shows the helpers in `qarp.endianness` for converting to and from
  integer labels.
* Values are **probabilities** (counts normalised by `n_shots`), not raw counts.

The result is also stored on the primitive itself:

In [ ]:
sampler.result

## Poke at it

Everything you just used is an ordinary Python object — inspect it:

In [ ]:
print("n_qubits:        ", bell.n_qubits)
print("command stream:  ", len(bell.flatten()), "commands")
print("first commands:  ", bell.flatten()[:3])
print("primitive target:", sampler.target)

## Where next

* **tutorial_01_blocks** — the Block model in full: composition, symbols,
  measurement, and the three conventions you must know.
* If you ever want the catalogue of every available block / primitive /
  algorithm, the `mwe_*.ipynb` notebooks in this folder are the reference
  gallery; the tutorials are the guided path.

---

*Source: [OpenQARP/openqarp, examples/tutorial_00_hello_qarp.ipynb](https://github.com/OpenQARP/openqarp/tree/main/examples), Apache License 2.0, Fujitsu Research of Europe.*